# ⚡ Heady Unified Latent Operating System
## Master Bootstrap Notebook — Node Alpha (Head)

**Architecture:** VSA State Machine → Ray Distributed Cluster → Tailscale Mesh → Redis IPC

**Execution Order:**
1. Mount Drive & sync monorepo
2. Install dependencies (torchhd, ray, redis)
3. Establish Tailscale mesh network
4. Initialize Redis IPC
5. Launch Ray cluster head
6. Boot VSA state machine
7. Worker node variant (for Nodes Beta/Gamma)

---
© 2026 Heady Systems LLC. Proprietary and Confidential.

In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 1: Persistent Storage & Monorepo Synchronization
# ══════════════════════════════════════════════════════════════════
import os, subprocess, json, time

# ─── Mount Google Drive (persistent state across restarts) ───
from google.colab import drive, userdata
drive.mount('/content/drive', force_remount=True)

WORKSPACE = '/content/drive/MyDrive/HeadyOS'
REPO_DIR  = os.path.join(WORKSPACE, 'heady-production')
STATE_DIR = os.path.join(WORKSPACE, 'state')       # persistent VSA memory
LOGS_DIR  = os.path.join(WORKSPACE, 'logs')        # persistent telemetry

for d in [WORKSPACE, STATE_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

# ─── Headless Git Sync ───
try:
    GITHUB_PAT = userdata.get('GITHUB_PAT')
except Exception:
    GITHUB_PAT = os.environ.get('GITHUB_PAT', '')

REPO_URL = f'https://{GITHUB_PAT}@github.com/HeadyMe/Heady-pre-production-9f2f0642.git'

if os.path.exists(os.path.join(REPO_DIR, '.git')):
    print('↻ Pulling latest from monorepo...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only', 'origin', 'main'],
                   capture_output=True, text=True)
    print('  ✓ Repo synced')
else:
    print('⬇ Cloning Heady monorepo...')
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR],
                   capture_output=True, text=True)
    print('  ✓ Repo cloned')

# ─── System manifest ───
manifest = {
    'node': 'alpha',
    'role': 'head',
    'boot_time': time.time(),
    'workspace': WORKSPACE,
    'repo': REPO_DIR,
    'gpu': subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader']).decode().strip() if os.path.exists('/usr/bin/nvidia-smi') else 'none',
}
with open(os.path.join(STATE_DIR, 'manifest.json'), 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'\n╔══ Heady Unified OS — Node Alpha ══╗')
print(f'║  GPU: {manifest["gpu"]}')
print(f'║  Workspace: {WORKSPACE}')
print(f'╚════════════════════════════════════╝')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 2: Install System Dependencies
# ══════════════════════════════════════════════════════════════════
import subprocess, sys

DEPS = [
    'torchhd',           # VSA / Hyperdimensional Computing
    'ray[default]',      # Distributed compute orchestration
    'redis',             # Inter-process communication
    'numpy',             # Tensor math
    'scikit-learn',      # Cosine similarity utilities
    'aiohttp',           # Async HTTP for bridge comms
]

print('📦 Installing dependencies...')
for dep in DEPS:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', dep],
        capture_output=True, text=True
    )
    status = '✓' if result.returncode == 0 else '✗'
    print(f'  {status} {dep}')

# Verify torch + GPU
import torch
gpu_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if gpu_available else 'CPU'
print(f'\n🔥 Compute: {gpu_name} | CUDA: {gpu_available}')
print(f'   PyTorch: {torch.__version__}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 3: Tailscale Mesh Network (Userspace Mode)
# ══════════════════════════════════════════════════════════════════
# Bridges isolated Colab VMs into a private encrypted mesh.
# Requires: TAILSCALE_AUTHKEY in Colab Secrets.
# Skip this cell if running single-node.
# ══════════════════════════════════════════════════════════════════
import subprocess, os

ENABLE_MESH = False  # Set True when running multi-node cluster

if ENABLE_MESH:
    try:
        from google.colab import userdata
        TS_KEY = userdata.get('TAILSCALE_AUTHKEY')
    except Exception:
        TS_KEY = os.environ.get('TAILSCALE_AUTHKEY', '')

    if not TS_KEY:
        print('⚠ TAILSCALE_AUTHKEY not set — skipping mesh setup')
        print('  Set it in Colab Secrets to enable multi-node clustering')
        TAILSCALE_IP = '127.0.0.1'
    else:
        print('🌐 Installing Tailscale (userspace mode)...')
        subprocess.run('curl -fsSL https://tailscale.com/install.sh | sh',
                       shell=True, capture_output=True)

        # Start in userspace — no kernel CAP_NET_ADMIN needed
        subprocess.run(
            f'tailscaled --tun=userspace-networking --socks5-server=localhost:1055 &',
            shell=True)
        import time; time.sleep(3)

        subprocess.run(
            f'tailscale up --authkey={TS_KEY} --hostname=heady-node-alpha --accept-routes',
            shell=True)

        try:
            TAILSCALE_IP = subprocess.check_output(
                ['tailscale', 'ip', '-4']).decode().strip()
            print(f'  ✓ Tailscale mesh active — IP: {TAILSCALE_IP}')
        except Exception:
            TAILSCALE_IP = '127.0.0.1'
            print('  ⚠ Could not get Tailscale IP, falling back to localhost')
else:
    TAILSCALE_IP = '127.0.0.1'
    print('ℹ Mesh networking disabled — single-node mode')
    print('  Set ENABLE_MESH = True for multi-node clustering')

print(f'\n📡 Node Alpha IP: {TAILSCALE_IP}')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 4: Redis IPC (Inter-Process Communication)
# ══════════════════════════════════════════════════════════════════
# Central nervous system for distributed state sharing.
# Agents publish/subscribe VSA state vectors via Redis Streams.
# ══════════════════════════════════════════════════════════════════
import subprocess, time

print('🔴 Starting Redis IPC server...')
subprocess.run(['apt-get', 'install', '-y', '-qq', 'redis-server'],
               capture_output=True)

# Bind to Tailscale IP so workers can connect (or localhost for single-node)
redis_conf = f'''
bind {TAILSCALE_IP} 127.0.0.1
port 6379
maxmemory 2gb
maxmemory-policy allkeys-lru
save ""
appendonly no
daemonize yes
'''

with open('/tmp/heady-redis.conf', 'w') as f:
    f.write(redis_conf)

subprocess.run(['redis-server', '/tmp/heady-redis.conf'])
time.sleep(1)

import redis
rds = redis.Redis(host='127.0.0.1', port=6379, decode_responses=True)

try:
    rds.ping()
    rds.set('heady:node:alpha:status', 'online')
    rds.set('heady:boot_time', str(time.time()))
    print(f'  ✓ Redis IPC operational on {TAILSCALE_IP}:6379')
    print(f'  ✓ Memory limit: 2GB LRU')
except redis.ConnectionError as e:
    print(f'  ✗ Redis failed: {e}')
    rds = None


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 5: Ray Distributed Computing Cluster (Head Node)
# ══════════════════════════════════════════════════════════════════
# Worker nodes connect via: ray.init(address='ray://<TAILSCALE_IP>:10001')
# ══════════════════════════════════════════════════════════════════
import ray, os

print('☀ Initializing Ray head node...')

if ray.is_initialized():
    ray.shutdown()

ray.init(
    num_cpus=os.cpu_count(),
    num_gpus=1,
    dashboard_host='0.0.0.0',
    runtime_env={'pip': ['torchhd', 'numpy']},
)

cluster = ray.cluster_resources()
print(f'  ✓ Ray cluster active')
print(f'  ✓ CPUs: {cluster.get("CPU", 0):.0f}')
print(f'  ✓ GPUs: {cluster.get("GPU", 0):.0f}')
print(f'  ✓ Memory: {cluster.get("memory", 0) / 1e9:.1f} GB')

if ENABLE_MESH:
    print(f'\n  Workers connect via: ray.init(address="ray://{TAILSCALE_IP}:10001")')
else:
    print(f'\n  Single-node mode — all compute is local')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 6: VSA State Machine — The Latent Operating System Core
# ══════════════════════════════════════════════════════════════════
# Replaces conditional JS branching with tensor-native operations.
# Logic primitives: Bind (⊗), Bundle (⊕), Similarity (⊙), Permute (ρ)
# ══════════════════════════════════════════════════════════════════
import torch, torchhd, numpy as np, json, time, os, ray

DIMS = 10000   # Hyperdimensional space (quasi-orthogonal at this range)
VSA  = 'MAP'   # Multiply-Add-Permute model
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'⚡ VSA Engine booting on {DEVICE} | D={DIMS} | Model={VSA}')

# ═══ Codebook: atomic concept hypervectors ═══
class VSACodebook:
    """Associative item memory — maps names to hypervectors."""
    def __init__(self, dims=DIMS, vsa=VSA, device=DEVICE):
        self.dims = dims
        self.vsa = vsa
        self.device = device
        self.items = {}       # name → hypervector
        self.labels = []      # ordered labels for batch similarity
        self._bank = None     # stacked tensor for fast lookup

    def add(self, name):
        if name not in self.items:
            hv = torchhd.random(1, self.dims, vsa=self.vsa, device=self.device).squeeze(0)
            self.items[name] = hv
            self.labels.append(name)
            self._bank = None  # invalidate cache
        return self.items[name]

    def get(self, name):
        return self.items.get(name, self.add(name))

    def lookup(self, query_hv, top_k=3):
        """Nearest-neighbor retrieval in the codebook."""
        if self._bank is None:
            self._bank = torch.stack(list(self.items.values()))
        sims = torchhd.cosine_similarity(query_hv.unsqueeze(0), self._bank).squeeze(0)
        topk = torch.topk(sims, min(top_k, len(self.labels)))
        return [(self.labels[i], sims[i].item()) for i in topk.indices]

    def __len__(self):
        return len(self.items)

# ═══ State Machine ═══
class VSAStateMachine:
    """Operating system logic via VSA — no if/else, pure tensor math."""
    def __init__(self):
        self.codebook = VSACodebook()
        self.history = []     # temporal state trace
        self.state_hv = None  # current system state

        # ─── Define role hypervectors (semantic slots) ───
        self.roles = {}
        for role in ['agent', 'action', 'target', 'context', 'priority',
                     'status', 'result', 'timestamp', 'confidence']:
            self.roles[role] = self.codebook.add(f'ROLE_{role}')

        # ─── Define system states ───
        for state in ['idle', 'processing', 'awaiting', 'executing',
                      'completed', 'failed', 'learning', 'optimizing']:
            self.codebook.add(f'STATE_{state}')

        # ─── Define action vocabulary ───
        for action in ['chat', 'embed', 'search', 'deploy', 'monitor',
                       'heal', 'optimize', 'learn', 'route', 'store',
                       'retrieve', 'analyze', 'synthesize', 'guard']:
            self.codebook.add(f'ACTION_{action}')

        # ─── Define agent identities ───
        for agent in ['conductor', 'buddy', 'brain', 'watchdog',
                      'optimizer', 'security', 'vector_ops']:
            self.codebook.add(f'AGENT_{agent}')

        self.state_hv = self.codebook.get('STATE_idle')
        print(f'  ✓ Codebook: {len(self.codebook)} concepts registered')

    def bind(self, role_name, value_name):
        """Bind a role to a value: role ⊗ value → new vector."""
        role_hv = self.roles.get(role_name, self.codebook.get(role_name))
        value_hv = self.codebook.get(value_name)
        return torchhd.bind(role_hv, value_hv)

    def bundle(self, *vectors):
        """Superpose multiple vectors: v1 ⊕ v2 ⊕ ... → unified state."""
        result = vectors[0]
        for v in vectors[1:]:
            result = torchhd.bundle(result, v)
        return result

    def transition(self, agent, action, target, context=None):
        """Execute a state transition via VSA tensor math.

        Instead of: if state == 'idle' and action == 'chat': state = 'processing'
        We compute:  new_state = bundle(bind(agent, A), bind(action, B), bind(target, C))
                     → similarity lookup against codebook → nearest state
        """
        bound_agent  = self.bind('agent', agent)
        bound_action = self.bind('action', action)
        bound_target = self.bind('target', target)

        parts = [bound_agent, bound_action, bound_target]
        if context:
            bound_ctx = self.bind('context', context)
            parts.append(bound_ctx)

        # Bundle all bindings into unified state
        new_state = self.bundle(*parts)

        # Apply temporal permutation to encode sequence order
        step = len(self.history)
        new_state = torchhd.permute(new_state, shifts=step % DIMS)

        # Record
        self.history.append({
            'step': step,
            'agent': agent,
            'action': action,
            'target': target,
            'context': context,
            'time': time.time(),
        })

        # Similarity lookup → nearest known state
        matches = self.codebook.lookup(new_state, top_k=5)
        self.state_hv = new_state

        return {
            'step': step,
            'nearest_concepts': matches,
            'state_norm': float(torch.norm(new_state)),
        }

    def query(self, query_name):
        """Instantaneous context retrieval — 3D vector space nearest-neighbor."""
        q = self.codebook.get(query_name)
        return self.codebook.lookup(q, top_k=5)

    def get_status(self):
        return {
            'codebook_size': len(self.codebook),
            'history_length': len(self.history),
            'dimensions': DIMS,
            'device': DEVICE,
            'model': VSA,
        }

# ─── Instantiate ───
vsa = VSAStateMachine()

# ─── Demo transitions ───
print('\n── VSA State Transitions ──')
demos = [
    ('AGENT_brain',     'ACTION_chat',      'user_query',     'STATE_idle'),
    ('AGENT_conductor', 'ACTION_route',     'AGENT_brain',    'STATE_processing'),
    ('AGENT_buddy',     'ACTION_learn',     'error_pattern',  'STATE_learning'),
    ('AGENT_watchdog',  'ACTION_guard',     'AGENT_buddy',    'STATE_monitoring'),
    ('AGENT_optimizer', 'ACTION_optimize',  'vector_space',   'STATE_optimizing'),
]

for agent, action, target, ctx in demos:
    t0 = time.perf_counter_ns()
    result = vsa.transition(agent, action, target, ctx)
    ns = time.perf_counter_ns() - t0
    top = result['nearest_concepts'][:3]
    top_str = ', '.join(f'{n}({s:.3f})' for n, s in top)
    print(f'  Step {result["step"]}: {agent}→{action}→{target}  [{ns/1000:.0f}µs]  nearest: {top_str}')

print(f'\n✓ VSA State Machine: {vsa.get_status()}')

# ─── Persist state to Drive ───
state_path = os.path.join(STATE_DIR, 'vsa_state.json')
with open(state_path, 'w') as f:
    json.dump({'status': vsa.get_status(), 'history': vsa.history}, f, indent=2)
print(f'✓ State persisted to {state_path}')

# ─── Publish to Redis IPC ───
if rds:
    rds.set('heady:vsa:status', json.dumps(vsa.get_status()))
    rds.set('heady:vsa:last_step', str(len(vsa.history)))
    print('✓ State published to Redis IPC')

print('\n⚡ Heady Unified Operating System is LIVE.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 7: Ray-Distributed VSA Operations
# ══════════════════════════════════════════════════════════════════
# Heavy tensor ops decorated with @ray.remote → auto-distributed
# across all GPUs in the cluster.
# ══════════════════════════════════════════════════════════════════
import ray, torch, torchhd, time, json

@ray.remote(num_gpus=0.5)
class DistributedVSAWorker:
    """GPU-accelerated VSA worker for heavy tensor operations."""

    def __init__(self, dims=10000, vsa='MAP'):
        self.dims = dims
        self.vsa = vsa
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        self.memory_bank = {}  # item memory
        print(f'VSAWorker initialized on {self.device}')

    def batch_embed(self, concepts):
        """Embed a batch of concepts into the hyperdimensional space."""
        vectors = torchhd.random(len(concepts), self.dims,
                                vsa=self.vsa, device=self.device)
        for i, name in enumerate(concepts):
            self.memory_bank[name] = vectors[i]
        return {'embedded': len(concepts), 'total': len(self.memory_bank)}

    def similarity_matrix(self, names):
        """Compute pairwise similarity between named concepts."""
        vecs = [self.memory_bank[n] for n in names if n in self.memory_bank]
        if len(vecs) < 2:
            return {'error': 'Need at least 2 concepts in memory'}
        bank = torch.stack(vecs)
        sims = torchhd.cosine_similarity(bank, bank)
        result = {}
        valid_names = [n for n in names if n in self.memory_bank]
        for i, n1 in enumerate(valid_names):
            result[n1] = {n2: round(sims[i][j].item(), 4)
                          for j, n2 in enumerate(valid_names)}
        return result

    def bundle_all(self):
        """Bundle all items in memory into a single superposition vector."""
        if not self.memory_bank:
            return None
        vecs = list(self.memory_bank.values())
        result = vecs[0]
        for v in vecs[1:]:
            result = torchhd.bundle(result, v)
        return float(torch.norm(result))

    def get_stats(self):
        return {
            'device': self.device,
            'memory_items': len(self.memory_bank),
            'dims': self.dims,
        }

# ─── Deploy worker ───
print('☀ Deploying distributed VSA worker...')
worker = DistributedVSAWorker.remote()

# ─── Batch embed system concepts ───
system_concepts = [
    'heady_brain', 'heady_conductor', 'heady_buddy', 'heady_watchdog',
    'vector_memory', 'self_awareness', 'deep_research', 'auto_heal',
    'user_query', 'api_request', 'error_pattern', 'optimization_target',
    'security_alert', 'health_check', 'state_transition', 'context_window',
    'latent_space', 'geometric_algebra', 'hyperdimensional', 'orchestration',
]

t0 = time.perf_counter()
embed_result = ray.get(worker.batch_embed.remote(system_concepts))
embed_ms = (time.perf_counter() - t0) * 1000
print(f'  ✓ Embedded {embed_result["embedded"]} concepts in {embed_ms:.1f}ms')

# ─── Compute similarity matrix ───
t0 = time.perf_counter()
sim_matrix = ray.get(worker.similarity_matrix.remote(
    ['heady_brain', 'heady_conductor', 'heady_buddy', 'vector_memory', 'latent_space']))
sim_ms = (time.perf_counter() - t0) * 1000
print(f'  ✓ Similarity matrix computed in {sim_ms:.1f}ms')

# ─── Bundle superposition ───
t0 = time.perf_counter()
bundle_norm = ray.get(worker.bundle_all.remote())
bundle_ms = (time.perf_counter() - t0) * 1000
print(f'  ✓ Global bundle norm: {bundle_norm:.2f} ({bundle_ms:.1f}ms)')

stats = ray.get(worker.get_stats.remote())
print(f'\n✓ Distributed Worker: {stats}')
print(f'✓ Cluster operational — ready for Nodes Beta & Gamma')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 8: Worker Node Bootstrap (Run on Nodes Beta / Gamma ONLY)
# ══════════════════════════════════════════════════════════════════
# Copy this cell to a separate notebook for worker nodes.
# Prerequisites: Tailscale mesh active, Node Alpha running.
# ══════════════════════════════════════════════════════════════════

WORKER_MODE = False  # Set True only on Beta/Gamma nodes

if WORKER_MODE:
    import ray, subprocess, os
    from google.colab import userdata

    # Get Node Alpha's Tailscale IP
    HEAD_IP = os.environ.get('HEAD_NODE_IP', '100.x.x.x')  # Set this!
    NODE_NAME = os.environ.get('NODE_NAME', 'beta')  # 'beta' or 'gamma'

    # Setup Tailscale
    TS_KEY = userdata.get('TAILSCALE_AUTHKEY')
    subprocess.run('curl -fsSL https://tailscale.com/install.sh | sh', shell=True)
    subprocess.run('tailscaled --tun=userspace-networking &', shell=True)
    import time; time.sleep(3)
    subprocess.run(
        f'tailscale up --authkey={TS_KEY} --hostname=heady-node-{NODE_NAME} --accept-routes',
        shell=True)

    # Connect to Ray head
    print(f'☀ Connecting to Ray head at {HEAD_IP}...')
    ray.init(address=f'ray://{HEAD_IP}:10001')
    print(f'  ✓ Node {NODE_NAME} joined cluster')
    print(f'  ✓ Resources: {ray.cluster_resources()}')
else:
    print('ℹ Worker mode disabled — this is the Head node.')
    print('  To run a worker, copy this cell to a separate Colab notebook,\n'
          '  set WORKER_MODE = True, and configure HEAD_IP.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# Cell 9: Interactive VSA Console
# ══════════════════════════════════════════════════════════════════
# Type commands to interact with the VSA state machine.
# ══════════════════════════════════════════════════════════════════

print('═══ Heady VSA Console ═══')
print('Commands:')
print('  transition <agent> <action> <target>  — execute state transition')
print('  query <concept>                       — nearest-neighbor lookup')
print('  status                                — system status')
print('  history                               — transition history')
print('  quit                                  — exit console')
print()

while True:
    try:
        cmd = input('VSA> ').strip()
        if not cmd:
            continue
        if cmd in ('quit', 'exit', 'q'):
            break

        parts = cmd.split()
        verb = parts[0].lower()

        if verb == 'status':
            import json
            print(json.dumps(vsa.get_status(), indent=2))

        elif verb == 'history':
            for h in vsa.history[-10:]:
                print(f'  [{h["step"]}] {h["agent"]}→{h["action"]}→{h["target"]}')

        elif verb == 'query' and len(parts) >= 2:
            concept = parts[1]
            matches = vsa.query(concept)
            for name, sim in matches:
                bar = '█' * int(abs(sim) * 30)
                print(f'  {name:30s} {sim:+.4f} {bar}')

        elif verb == 'transition' and len(parts) >= 4:
            agent, action, target = parts[1], parts[2], parts[3]
            ctx = parts[4] if len(parts) > 4 else None
            result = vsa.transition(agent, action, target, ctx)
            print(f'  Step {result["step"]} | norm={result["state_norm"]:.2f}')
            for name, sim in result['nearest_concepts'][:5]:
                print(f'    {name}: {sim:+.4f}')

        else:
            print(f'  Unknown: {cmd}')

    except (EOFError, KeyboardInterrupt):
        break
    except Exception as e:
        print(f'  Error: {e}')

print('VSA console closed.')
